# Getting Financial Data - Pandas Datareader

### Step 1. Import the necessary libraries

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

DATA_ROOT = Path(os.environ.get("PANDAS_DATA_ROOT", r"../pandas_exercises"))

def data_file(name):
    direct = DATA_ROOT / name
    if direct.is_file():
        return direct
    matches = list(DATA_ROOT.rglob(name)) if DATA_ROOT.exists() else []
    if len(matches) != 1:
        raise FileNotFoundError(f"Set DATA_ROOT to your extracted data folder. Expected one {name}; found {len(matches)}.")
    return matches[0]

USE_LOCAL_CSV = True
API_KEY = os.environ.get('ALPHAVANTAGE_API_KEY')

### Step 2. Create your time range (start and end variables). The start date should be 01/01/2015 and the end should today (whatever your today is).

In [ ]:
start = pd.Timestamp('2015-01-01')
end = pd.Timestamp.today().normalize()

### Step 3. Get an API key for one of the APIs that are supported by Pandas Datareader, preferably for AlphaVantage.

If you do not have an API key for any of the supported APIs, it is easiest to get one for [AlphaVantage](https://www.alphavantage.co/support/#api-key). (Note that the API key is shown directly after the signup. You do *not* receive it via e-mail.)

(For a full list of the APIs that are supported by Pandas Datareader, [see here](https://pydata.github.io/pandas-datareader/readers/index.html). As the APIs are provided by third parties, this list may change.)

In [ ]:
# Obtain a personal key at https://www.alphavantage.co/support/#api-key
# Set ALPHAVANTAGE_API_KEY outside the notebook; never commit the key.
API_KEY = os.environ.get('ALPHAVANTAGE_API_KEY')

### Step 4. Use Pandas Datarader to read the daily time series for the Apple stock (ticker symbol AAPL) between 01/01/2015 and today, assign it to df_apple and print it.

In [ ]:
def get_stock(symbol):
    if USE_LOCAL_CSV:
        frame = pd.read_csv(data_file(f'{symbol}.csv'))
        frame.columns = frame.columns.str.strip().str.lower()
        date_col = 'date' if 'date' in frame else 'timestamp'
        frame = frame.set_index(date_col)
    else:
        from pandas_datareader.av.time_series import AVTimeSeriesReader
        if not API_KEY:
            raise ValueError('Set ALPHAVANTAGE_API_KEY in your environment.')
        frame = AVTimeSeriesReader(symbol, function='TIME_SERIES_DAILY', start=start, end=end, api_key=API_KEY).read()
    frame.index = pd.to_datetime(frame.index)
    frame.index.name = 'Date'
    frame = frame.sort_index().loc[start:end].copy()
    if frame.empty or frame.index.min() > start + pd.Timedelta(days=7):
        raise ValueError(f'{symbol}: full historical data beginning in January 2015 is required.')
    frame['volume'] = pd.to_numeric(frame['volume'])
    return frame

df_apple = get_stock('AAPL')
df_apple

### Step 5. Add a new column "stock" to the dataframe and add the ticker symbol

In [ ]:
df_apple['stock'] = 'AAPL'
df_apple

### Step 6. Repeat the two previous steps for a few other stocks, always creating a new dataframe: Tesla, IBM and Microsoft. (Ticker symbols TSLA, IBM and MSFT.)

In [ ]:
df_tesla = get_stock('TSLA').assign(stock='TSLA')
df_ibm = get_stock('IBM').assign(stock='IBM')
df_microsoft = get_stock('MSFT').assign(stock='MSFT')

### Step 7. Combine the four separate dataFrames into one combined dataFrame df that holds the information for all four stocks

In [ ]:
df = pd.concat([df_apple, df_tesla, df_ibm, df_microsoft])
df

### Step 8. Shift the stock column into the index (making it a multi-level index consisting of the ticker symbol and the date).

In [ ]:
df = df.reset_index().set_index(['stock', 'Date']).sort_index()
df

### Step 7. Create a dataFrame called vol, with the volume values.

In [ ]:
vol = df[['volume']].copy()
vol

### Step 8. Aggregate the data of volume to weekly.
Hint: Be careful to not sum data from the same week of 2015 and other years.

In [ ]:
weekly_vol = vol.groupby([pd.Grouper(level='stock'), pd.Grouper(level='Date', freq='W-SUN')]).sum(min_count=1)
weekly_vol

### Step 9. Find all the volume traded in the year of 2015

In [ ]:
volume_2015 = vol.loc[vol.index.get_level_values('Date').year == 2015].groupby(level='stock').sum()
display(volume_2015)
volume_2015.sum()